## HW 3

In this assignment, we will learn about locality sensitive hashing in Apache Spark.

Start by running the code below if you are using Google Colab

In [218]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://www-eu.apache.org/dist/spark/spark-2.4.5/spark-2.4.5-bin-hadoop2.7.tgz
!tar xf spark-2.4.5-bin-hadoop2.7.tgz

In [219]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-2.4.5-bin-hadoop2.7"

In [220]:
!pip install -q findspark
!pip install pyspark

In [221]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [222]:
from pyspark.sql import SparkSession

In [223]:
APP_NAME = "HW3"

In [224]:
spark = SparkSession.builder.appName(APP_NAME).getOrCreate()

In [225]:
spark

1. Load the wikipedia links dataset provided in this assignment as a dataframe.

In [226]:
import os

CSV_PATH = "/content/wikipedia_links (1).csv"
for p in ["wikipedia_links-1-2.csv", "wikipedia_links.csv"]:
    if os.path.exists(p):
        CSV_PATH = p
        break
if CSV_PATH is None:
    CSV_PATH = "wikipedia_links-1-2.csv"  # <- adjust to your Drive path if needed

wiki = spark.read.csv(CSV_PATH, header=True, inferSchema=True)

print("Row count:", wiki.count())
wiki.printSchema()
wiki.show(10, truncate=False)

Row count: 60000
root
 |-- link: string (nullable = true)

+--------------------------------------------------------------+
|link                                                          |
+--------------------------------------------------------------+
|http://en.wikipedia.org/wiki/Lungotevere_dei_Mellini          |
|http://en.wikipedia.org/wiki/South_Rowan_High_School          |
|http://en.wikipedia.org/wiki/Haikou_College_of_Economics      |
|http://en.wikipedia.org/wiki/Saylesville,_Rhode_Island        |
|http://en.wikipedia.org/wiki/Wallington,_New_Jersey           |
|http://en.wikipedia.org/wiki/Saint_Andrew's_Chapel            |
|http://en.wikipedia.org/wiki/Nemperor_Records                 |
|http://en.wikipedia.org/wiki/Kepler-37b                       |
|http://en.wikipedia.org/wiki/Ecomuseo_della_Montagna_Pistoiese|
|http://en.wikipedia.org/wiki/Virtual_Network_Computing        |
+--------------------------------------------------------------+
only showing top 10 rows


2. We would like to keep the name of the Wikipedia entry only. Parse the links on the '/' symbol and keep only the last part (you may use PySpark functions for string splitting).

In [227]:
from pyspark.sql import functions as F

wiki = wiki.withColumn("title", F.element_at(F.split(F.col("link"), "/"), -1))

wiki.select("link", "title").show(10, truncate=False)


+--------------------------------------------------------------+---------------------------------+
|link                                                          |title                            |
+--------------------------------------------------------------+---------------------------------+
|http://en.wikipedia.org/wiki/Lungotevere_dei_Mellini          |Lungotevere_dei_Mellini          |
|http://en.wikipedia.org/wiki/South_Rowan_High_School          |South_Rowan_High_School          |
|http://en.wikipedia.org/wiki/Haikou_College_of_Economics      |Haikou_College_of_Economics      |
|http://en.wikipedia.org/wiki/Saylesville,_Rhode_Island        |Saylesville,_Rhode_Island        |
|http://en.wikipedia.org/wiki/Wallington,_New_Jersey           |Wallington,_New_Jersey           |
|http://en.wikipedia.org/wiki/Saint_Andrew's_Chapel            |Saint_Andrew's_Chapel            |
|http://en.wikipedia.org/wiki/Nemperor_Records                 |Nemperor_Records                 |
|http://en

# Question 2: Keep only the Wikipedia entry name (last part after '/')
from pyspark.sql import functions as F

wiki = wiki.withColumn('name', F.element_at(F.split(F.col('link'), '/'), -1)) \
           .select('name')

wiki.show(10, truncate=False)

3. Add an ID column that will be a sequential integer between 1 and len(wiki) in our Wikipedia links table. You may order the column using any other column of your choice.

In [228]:
from pyspark.sql.window import Window

wiki = wiki.withColumn("id", F.row_number().over(Window.orderBy("title")))

wiki.select("id", "title", "link").show(10, truncate=False)
print("Min id:", wiki.agg(F.min("id")).first()[0],
      "Max id:", wiki.agg(F.max("id")).first()[0],
      "Row count:", wiki.count())


+---+---------------------+--------------------------------------------------+
|id |title                |link                                              |
+---+---------------------+--------------------------------------------------+
|1  |NULL                 |NULL                                              |
|2  |!!!                  |http://en.wikipedia.org/wiki/!!!                  |
|3  |!!!                  |http://en.wikipedia.org/wiki/!!!                  |
|4  |!Oka_Tokat           |http://en.wikipedia.org/wiki/!Oka_Tokat           |
|5  |!PAUS3               |http://en.wikipedia.org/wiki/!PAUS3               |
|6  |!PAUS3               |http://en.wikipedia.org/wiki/!PAUS3               |
|7  |!PAUS3               |http://en.wikipedia.org/wiki/!PAUS3               |
|8  |!PAUS3               |http://en.wikipedia.org/wiki/!PAUS3               |
|9  |!Women_Art_Revolution|http://en.wikipedia.org/wiki/!Women_Art_Revolution|
|10 |$_(film)             |http://en.wikipedia.org/w

4. Clean up the data by removing missing data.

In [229]:
before = wiki.count()
wiki = wiki.na.drop(subset=["link", "title"])
wiki = wiki.filter((F.col("title").isNotNull()) & (F.trim(F.col("title")) != ""))

# Re-sequence ids so they stay gap-free (1..N) after dropping rows.
wiki = wiki.withColumn("id", F.row_number().over(Window.orderBy("title")))

after = wiki.count()
print(f"Dropped {before - after} rows with missing link/title. Clean rows: {after}")
wiki.show(5, truncate=False)

Dropped 1 rows with missing link/title. Clean rows: 59999
+---------------------------------------+----------+---+
|link                                   |title     |id |
+---------------------------------------+----------+---+
|http://en.wikipedia.org/wiki/!!!       |!!!       |1  |
|http://en.wikipedia.org/wiki/!!!       |!!!       |2  |
|http://en.wikipedia.org/wiki/!Oka_Tokat|!Oka_Tokat|3  |
|http://en.wikipedia.org/wiki/!PAUS3    |!PAUS3    |4  |
|http://en.wikipedia.org/wiki/!PAUS3    |!PAUS3    |5  |
+---------------------------------------+----------+---+
only showing top 5 rows


5. Tokenize the words by splitting on the underscore symbol, count vectorize using a vocab size of 1,000,000, and create feature vectors using the count vectors. Your final result should be a dataframe containing the ID we previously generated and the feature vectors.

In [230]:
from pyspark.ml.feature import CountVectorizer

tokenized = (wiki.withColumn("words", F.split(F.lower(F.col("title")), "_"))
                 .withColumn("words", F.array_remove(F.col("words"), "")))

cv = CountVectorizer(inputCol="words", outputCol="features", vocabSize=1000000)
cv_model = cv.fit(tokenized)
featurized = cv_model.transform(tokenized)

# Final result requested by the assignment: ID + feature vectors only.
features = featurized.filter(F.size(F.col("words")) > 0).select("id", "features")

# Title-enriched copy, used only to make Part 6's output readable.
features_with_title = featurized.filter(F.size(F.col("words")) > 0) \
                                 .select("id", "title", "features")

print("Vocabulary size learned:", len(cv_model.vocabulary))
features.show(10, truncate=False)

Vocabulary size learned: 58014
+---+---------------------------------------+
|id |features                               |
+---+---------------------------------------+
|1  |(58014,[9258],[1.0])                   |
|2  |(58014,[9258],[1.0])                   |
|3  |(58014,[38535,58000],[1.0,1.0])        |
|4  |(58014,[5581],[1.0])                   |
|5  |(58014,[5581],[1.0])                   |
|6  |(58014,[5581],[1.0])                   |
|7  |(58014,[5581],[1.0])                   |
|8  |(58014,[268,1101,46720],[1.0,1.0,1.0]) |
|9  |(58014,[16,15356],[1.0,1.0])           |
|10 |(58014,[229,41441,45367],[1.0,1.0,1.0])|
+---+---------------------------------------+
only showing top 10 rows


6. In the last part of the assignment, use the MinHashLSH model to fit our feature vectors and then find all pairs using a threshold of 0.3.

In [231]:
from pyspark.ml.feature import MinHashLSH

features_with_title = features_with_title.cache()

mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
mh_model = mh.fit(features_with_title)

similar_pairs = mh_model.approxSimilarityJoin(
        features_with_title, features_with_title, 0.3, distCol="JaccardDistance"
    ).select(
        F.col("datasetA.id").alias("idA"),
        F.col("datasetA.title").alias("titleA"),
        F.col("datasetB.id").alias("idB"),
        F.col("datasetB.title").alias("titleB"),
        "JaccardDistance",
    ) \
    .filter(F.col("idA") < F.col("idB")) \
    .orderBy("JaccardDistance")

print("Number of similar pairs found (distance <= 0.3):", similar_pairs.count())
similar_pairs.show(20, truncate=False)

Number of similar pairs found (distance <= 0.3): 2318
+---+------------------------------+---+------------------------------+---------------+
|idA|titleA                        |idB|titleB                        |JaccardDistance|
+---+------------------------------+---+------------------------------+---------------+
|1  |!!!                           |2  |!!!                           |0.0            |
|4  |!PAUS3                        |5  |!PAUS3                        |0.0            |
|5  |!PAUS3                        |6  |!PAUS3                        |0.0            |
|4  |!PAUS3                        |6  |!PAUS3                        |0.0            |
|6  |!PAUS3                        |7  |!PAUS3                        |0.0            |
|5  |!PAUS3                        |7  |!PAUS3                        |0.0            |
|4  |!PAUS3                        |7  |!PAUS3                        |0.0            |
|16 |%22P%22_Is_for_Peril          |17 |%22P%22_Is_for_Peril      